# Лабораторная работа 1 — рабочий ноутбук

**ФИО:** Гончар Александр Сергеевич
**Файлы данных:** `data/clients.csv`, `data/transactions.csv`

Этот ноутбук — чистовой рабочий документ, а не черновик. Он должен
исполняться сверху вниз без ошибок на выданном вам **сыром** файле
и на выходе давать очищенную таблицу и `cleaning_log.csv`.

Разделы ниже обязательны, порядок менять не нужно. Внутри раздела
добавляйте столько ячеек, сколько нужно. Если какой-то раздел у вас
пустой — напишите в нём одной строкой, почему.

## 0. Окружение и настройки

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 50)

RAW_CLIENTS = "data/clients.csv"
RAW_TXN = "data/transactions.csv"

## 1. Загрузка сырых данных

Файлы выгружены разными системами. Убедитесь, что они прочитались
**правильно**: кодировка, разделитель, десятичный разделитель, типы,
даты, идентификаторы.

Пробуем прочитать
файл с разными параметрами. Неверная кодировка дает `UnicodeDecodeError`,
неверный разделитель дает один столбец вместо нескольких.

In [2]:
for path in [RAW_CLIENTS, RAW_TXN]:
    for enc in ["utf-8", "cp1251"]:
        for sep in [",", ";"]:
            try:
                df = pd.read_csv(path, encoding=enc, sep=sep)
                print(path, enc, repr(sep), "- столбцов:", df.shape[1])
            except UnicodeDecodeError:
                print(path, enc, repr(sep), "- ошибка кодировки")

data/clients.csv utf-8 ',' - столбцов: 16
data/clients.csv utf-8 ';' - столбцов: 1
data/clients.csv cp1251 ',' - ошибка кодировки
data/clients.csv cp1251 ';' - ошибка кодировки
data/transactions.csv utf-8 ',' - ошибка кодировки
data/transactions.csv utf-8 ';' - ошибка кодировки
data/transactions.csv cp1251 ',' - столбцов: 1
data/transactions.csv cp1251 ';' - столбцов: 9


`clients.csv` кодировка UTF-8, разделитель ',' (16 столбцов). `transactions.csv` кодировка cp1251,
разделитель ';' (9 столбцов). Но то, что файл открылся в cp1251, еще не значит, что кодировка верная так как
в cp1251 откроется почти любой файл, просто вместо букв может получиться мусор.
Поэтому дальше будем смотреть на сам текст.

Смотрим первые строки transactions и проверяем, что русские слова читаются.

In [3]:
pd.read_csv(RAW_TXN, encoding="cp1251", sep=";", nrows=5)

,txn_id,client_id,txn_datetime,amount,currency,category,mcc_code,channel,status
0,990102,C003921,2023-11-24 09:00:10,"1018,07",RUB,Кафе/рестораны,5812,Карта,успешно
1,990012,C002226,2023-11-25 17:53:53,"454,49",RUB,Продукты,5411,Карта,успешно
2,990027,C000544,2023-11-27 13:30:37,"1194,49",RUB,Продукты,5411,Онлайн-банк,успешно
3,990169,C002702,2023-11-27,"755,99",RUB,Онлайн-сервисы,5815,Карта,успешно
4,990082,C001351,2023-12-12,"1911,83",RUB,Продукты,5411,Карта,успешно


Категории, каналы и статусы читаются как нормальные русские слова,
значит кодировка cp1251 выбрана верно.
Также в amount дробная часть отделена запятой.

Читаем `transactions.csv` без указания десятичного разделителя и смотрим тип amount.

In [4]:
txn_try = pd.read_csv(RAW_TXN, encoding="cp1251", sep=";")
print(txn_try["amount"].dtype, txn_try["amount"].head(3).tolist())

str ['1018,07', '454,49', '1194,49']


Видим, что amount прочитался как текст, так как pandas по умолчанию ждет точку, а в файле запятая. Поэтому указываем `decimal=","`.

In [5]:
txn_try = pd.read_csv(RAW_TXN, encoding="cp1251", sep=";", decimal=",")
print(txn_try["amount"].dtype, txn_try["amount"].head(5).tolist())

float64 [1018.07, 454.49, 1194.49, 755.99, 1911.83]


С `decimal=","` amount стал числом. pandas делает столбец числовым только если смог перевести каждое значение, так что все суммы прочитались правильно.

Итоговое чтение обоих файлов с найденными параметрами.

`client_id`, `txn_id`, `branch_code`, `mcc_code` читаем как текст. Это метки, а не числа и складывать их бессмысленно. У `branch_code` вида '00409' при чтении числом потерялись бы ведущие нули.
Даты pandas сам оставляет текстом, форматы дат в файлах разные, в разделе 3.1 этим займемся.

In [6]:
clients_raw = pd.read_csv(RAW_CLIENTS, dtype={"client_id": str, "branch_code": str})
txn_raw = pd.read_csv(RAW_TXN, encoding="cp1251", sep=";", decimal=",",
                      dtype={"txn_id": str, "client_id": str, "mcc_code": str})

Проверяем результат: `info()` показывает число строк, тип каждого столбца и сколько в нем непустых значений.

In [7]:
clients_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 5690 entries, 0 to 5689
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   client_id           5690 non-null   str    
 1   birth_date          5690 non-null   str    
 2   gender              5690 non-null   str    
 3   city                5690 non-null   str    
 4   segment             5690 non-null   str    
 5   loyalty_tier        5690 non-null   str    
 6   card_type           5690 non-null   str    
 7   registration_date   5690 non-null   str    
 8   card_open_date      5690 non-null   str    
 9   income_source       5690 non-null   str    
 10  monthly_income      5204 non-null   float64
 11  has_mobile_app      5690 non-null   str    
 12  app_last_login      3525 non-null   str    
 13  satisfaction_score  2331 non-null   float64
 14  branch_code         5690 non-null   str    
 15  marketing_opt_in    5690 non-null   str    
dtypes: float64(2), st

In [8]:
txn_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 626169 entries, 0 to 626168
Data columns (total 9 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   txn_id        626169 non-null  str    
 1   client_id     626169 non-null  str    
 2   txn_datetime  626169 non-null  str    
 3   amount        626169 non-null  float64
 4   currency      626169 non-null  str    
 5   category      626169 non-null  str    
 6   mcc_code      626169 non-null  str    
 7   channel       626169 non-null  str    
 8   status        626169 non-null  str    
dtypes: float64(1), str(8)
memory usage: 43.0 MB


Итог загрузки:

`clients.csv` – 5690 строк и 16 столбцов, `transactions.csv` – 626169 строк и 9 столбцов.

Столбцы `amount` и `monthly_income` прочитались как числа (`float64`), а id и даты остались текстом. По `Non-Null Count` видно пропуски в `monthly_income`, `app_last_login` и `satisfaction_score`, их разберем в разделе 3.3.

`satisfaction_score` тоже получился `float64`, хотя оценки целые от 1 до 5, это из-за пропусков. 

## 2. Аудит качества

Здесь вы ничего не исправляете — только находите и измеряете.
Для каждой метрики качества заранее определите знаменатель.

### 2.1. Объём, типы, ключи, единица наблюдения

Собираем короткий паспорт обеих таблиц: объем, типы, ключ и единица наблюдения, то есть что означает одна строка.
По описанию в `clients.csv` одна строка – это один клиент с ключом `client_id`, а в `transactions.csv` одна строка – одна операция с ключом `txn_id`.
Если так, ключ должен быть уникальным. Доля лишних строк считается от числа строк таблицы.

In [9]:
passport = pd.DataFrame({
    "строк": [len(clients_raw), len(txn_raw)],
    "столбцов": [clients_raw.shape[1], txn_raw.shape[1]],
    "числовых столбцов": [clients_raw.select_dtypes("number").shape[1], txn_raw.select_dtypes("number").shape[1]],
    "ключ": ["client_id", "txn_id"],
    "уникальных ключей": [clients_raw["client_id"].nunique(), txn_raw["txn_id"].nunique()],
}, index=["clients.csv", "transactions.csv"])
passport["лишних строк"] = passport["строк"] - passport["уникальных ключей"]
passport["доля лишних, %"] = (passport["лишних строк"] / passport["строк"] * 100).round(1)
passport

,строк,столбцов,числовых столбцов,ключ,уникальных ключей,лишних строк,"доля лишних, %"
clients.csv,5690,16,2,client_id,5635,55,1.0
transactions.csv,626169,9,1,txn_id,624669,1500,0.2


В `clients.csv` 16 столбцов, из них числовых только 2 (`monthly_income` и `satisfaction_score`), в `transactions.csv` из 9 столбцов числовой один – `amount`. Остальное текст.

Ключи не уникальны. В `clients.csv` 5690 строк, но разных `client_id` только 5635, лишних строк 55.
В `transactions.csv` лишних строк 1500.
Значит один клиент или одна операция могут встречаться несколько раз, и единица наблюдения не совпадает с описанием.

### 2.2. Дубли

Начинаем с `clients.csv`. Повтор `client_id` бывает двух видов: полная копия строки (все поля совпадают) или разные записи с одним id.
Первое – техническое дублирование выгрузки, второе – противоречие: непонятно, какая запись верная.

In [10]:
print("лишних строк по client_id:", clients_raw["client_id"].duplicated().sum(), "из", len(clients_raw))
print("из них полных копий строк:", clients_raw.duplicated().sum())

лишних строк по client_id: 55 из 5690
из них полных копий строк: 40


Из 55 повторов 40 – точные копии, их можно просто убрать. А вот у 15 клиентов записи разные, дальше посмотрим, какие поля не совпадают.

In [11]:
clients_dup = clients_raw[clients_raw["client_id"].duplicated(keep=False)]
clients_dup_unique = clients_dup.drop_duplicates()
clients_conflict = clients_dup_unique[clients_dup_unique["client_id"].duplicated(keep=False)]
print("client_id с разными данными в повторах:", clients_conflict["client_id"].nunique())

diff_cols = (clients_conflict.groupby("client_id").nunique(dropna=False) > 1).sum()
diff_cols[diff_cols > 0]

client_id с разными данными в повторах: 15


city       14
segment    10
dtype: int64

Различия только в двух полях: `city` (14 клиентов) и `segment` (10 клиентов). По числам не понять, это опечатки в написании города или разные записи, поэтому смотрим на примеры. 

In [12]:
clients_conflict.sort_values("client_id")[["client_id", "city", "segment"]]

,client_id,city,segment
3322,C001083,Новосибирск,Премиальный
2067,C001083,Новосибирск,Зарплатный
3937,C001632,ЕКАТЕРИНБУРГ,Private
4864,C001632,Санкт-Петербург,Массовый
4711,C002094,Новосибирск,Зарплатный
786,C002094,г.Екатеринбург,Премиальный
1955,C002355,Нижний Новгород,Премиальный
4763,C002355,Казань,Массовый
4369,C002475,г.Екатеринбург,Премиальный
1601,C002475,Казань,Зарплатный


Названия городов в данных записаны по-разному: один и тот же город встречается с разным регистром, сокращениями и с «г.» или без. Даже если считать разные написания одним городом, у большинства из 15 клиентов две записи все равно указывают на разные города, а часто и на разные сегменты. Для этих клиентов настоящие город и сегмент неизвестны.

Один человек может быть записан под разными `client_id`. Проверяем по признакам, которые у человека не меняются: дата рождения, пол, дата прихода в банк и дата открытия карты. Берем по одной строке на `client_id`, чтобы уже найденные повторы не мешали.

In [13]:
stable = ["birth_date", "gender", "registration_date", "card_open_date"]
one_per_id = clients_raw.drop_duplicates("client_id")
print("одних и тех же людей под разными client_id:", one_per_id.duplicated(subset=stable).sum())

одних и тех же людей под разными client_id: 0


Под разными `client_id` нет клиентов с одинаковыми датой рождения, полом и датами регистрации и открытия карты.

Теперь `transactions.csv`, проверяем повторы `txn_id` так же.

In [14]:
print("лишних строк по txn_id:", txn_raw["txn_id"].duplicated().sum(), "из", len(txn_raw))
print("из них полных копий строк:", txn_raw.duplicated().sum())

лишних строк по txn_id: 1500 из 626169
из них полных копий строк: 1259


Из 1500 повторов 1259 – точные копии строк. У остальных 241 записи с одним `txn_id` чем-то отличаются, посмотрим чем они будут отличаться.

In [15]:
txn_dup = txn_raw[txn_raw["txn_id"].duplicated(keep=False)]
txn_dup_unique = txn_dup.drop_duplicates()
txn_conflict = txn_dup_unique[txn_dup_unique["txn_id"].duplicated(keep=False)]
print("txn_id с разными данными в повторах:", txn_conflict["txn_id"].nunique())

txn_diff_cols = (txn_conflict.groupby("txn_id").nunique(dropna=False) > 1).sum()
txn_diff_cols[txn_diff_cols > 0]

txn_id с разными данными в повторах: 241


txn_datetime    241
dtype: int64

Различается только по полю `txn_datetime`. Проверяем, как именно они различаются.

In [16]:
pairs = txn_conflict.groupby("txn_id")["txn_datetime"]
print("пар, где дата одна и та же:", (pairs.agg(lambda s: s.str[:10].nunique()) == 1).sum())
print("пар, где в одной записи только дата без времени:", (pairs.agg(lambda s: s.str.len().min()) == 10).sum())
txn_conflict.sort_values("txn_id").head(4)

пар, где дата одна и та же: 241
пар, где в одной записи только дата без времени: 241


,txn_id,client_id,txn_datetime,amount,currency,category,mcc_code,channel,status
208,100202,C003891,2024-01-01 11:56:08,495.24,RUB,Продукты,5411,Карта,успешно
209,100202,C003891,2024-01-01,495.24,RUB,Продукты,5411,Карта,успешно
3673,103657,C002052,2024-01-05,783.06,RUB,Продукты,5411,Карта,успешно
3674,103657,C002052,2024-01-05 14:51:41,783.06,RUB,Продукты,5411,Карта,успешно


Во всех 241 парах дата одна и та же, просто в одной записи время есть, а в другой только дата. Остальные поля, включая сумму, совпадают. Значит это одна и та же операция, записанная дважды. У нас выходит 1259 точных копий и 241 копия, которые отличаются только форматом даты.

### 2.3. Соответствие данных описанию полей из задания

Сначала категориальные поля. Для тех, у которых в описании задания перечислены значения, проверяем, нет ли в данных других значений. Считаем по сырым строкам, дубли еще не убраны.

In [17]:
expected = {
    "segment": ["Массовый", "Зарплатный", "Премиальный", "Private"],
    "loyalty_tier": ["Базовый", "Серебро", "Золото", "Платина"],
    "card_type": ["Дебетовая", "Кредитная", "Премиальная"],
}
for col, values in expected.items():
    extra = clients_raw.loc[~clients_raw[col].isin(values), col]
    print(col, "| значений не из описания:", len(extra))

expected_txn = {
    "channel": ["Карта", "Токен", "СБП", "Онлайн-банк"],
    "status": ["успешно", "отклонено", "возврат"],
}
for col, values in expected_txn.items():
    extra = txn_raw.loc[~txn_raw[col].isin(values), col]
    print(col, "| значений не из описания:", len(extra))

segment | значений не из описания: 0
loyalty_tier | значений не из описания: 0
card_type | значений не из описания: 0
channel | значений не из описания: 0
status | значений не из описания: 0


В `segment`, `loyalty_tier`, `card_type`, `channel` и `status` все значения совпадают с описанием.

Для остальных полей значения в описании не перечислены, поэтому смотрим на них через `value_counts()`.

In [18]:
for col in ["gender", "has_mobile_app", "marketing_opt_in", "income_source"]:
    display(clients_raw[col].value_counts(dropna=False))

gender
Ж    2859
М    2831
Name: count, dtype: int64

has_mobile_app
Да     3525
Нет    2165
Name: count, dtype: int64

marketing_opt_in
Да     5022
Нет     668
Name: count, dtype: int64

income_source
Зарплата в нашем банке     2547
Зарплата в другом банке    1599
Не указан                  1544
Name: count, dtype: int64

`gender`, `has_mobile_app` и `marketing_opt_in` принимают по два значения, тут все чисто.
В `income_source` есть значение `Не указан` – 1544 строки из 5690. Это пропуск, просто вместо пустой ячейки в ней стоит текст, поэтому `info()` считает эти строки заполненными, хотя источник дохода в них неизвестен.

В описании сказано, что `category` – одна из девяти категорий. Проверяем.

In [19]:
txn_raw["category"].value_counts()

category
Продукты               164934
Транспорт               65912
Одежда и обувь          65275
АЗС                     58023
Кафе и рестораны        51038
Развлечения и досуг     48162
Аптеки                  45391
Онлайн-сервисы          44112
Связь и интернет        29305
Связь и Интернет        19592
Кафе/рестораны          17226
кафе и рестораны        17199
Name: count, dtype: int64

Категорий 12, а не 9. Лишние появились из-за разного написания. Если их объединить, то у нас получится как раз 9.

Про валюту в описании ничего не сказано, смотрим, какие значения есть.

In [20]:
txn_raw["currency"].value_counts()

currency
RUB    623636
EUR      1283
USD      1250
Name: count, dtype: int64

Кроме рублей есть операции в `EUR` и `USD`. Суммы в разных валютах нельзя просто складывать, а курсов в данных нет.

По описанию `city` – город проживания. Смотрим, как записаны города. 

In [21]:
print("разных значений city:", clients_raw["city"].nunique())
clients_raw["city"].value_counts()

разных значений city: 17


city
Москва                1129
Новосибирск            681
Санкт-Петербург        657
Казань                 563
Нижний Новгород        534
Екатеринбург           526
г. Новосибирск         224
МОСКВА                 183
г. Москва              178
 москва                171
Н. Новгород            162
казань                 142
г. Санкт-Петербург     131
Санкт Петербург        113
СПб                    109
г.Екатеринбург          98
ЕКАТЕРИНБУРГ            89
Name: count, dtype: int64

Разных значений 17, а городов на самом деле 6 .Один и тот же город записан по-разному: разный регистр, приставка, пробел в начале, сокращения, запись без дефиса.

По описанию `mcc_code` соответствует категории. Смотрим таблицу сопряженности: сколько операций в каждой паре категории и кода.

In [22]:
pd.crosstab(txn_raw["category"], txn_raw["mcc_code"])

mcc_code,4111,4814,5411,5541,5651,5812,5815,5912,7832
category,,,,,,,,,
АЗС,29,20,23,57854,22,24,15,17,19
Аптеки,18,15,7,19,11,14,21,45277,9
Кафе и рестораны,12,12,18,16,18,50913,22,10,17
Кафе/рестораны,3,9,7,5,4,17184,8,3,3
Одежда и обувь,22,24,33,17,65106,18,22,17,16
Онлайн-сервисы,11,9,14,18,16,8,44013,10,13
Продукты,53,61,164482,53,58,53,66,57,51
Развлечения и досуг,20,15,16,16,19,14,8,14,48040
Связь и Интернет,8,19540,4,8,8,7,7,7,3


У каждой категории есть свой основной код, в нем почти все операции, а разные написания одной категории имеют один и тот же код. Но в каждой строке есть небольшие числа и в чужих столбцах – это операции, где код не совпадает с категорией. Считаем, сколько их.

In [23]:
main_mcc = txn_raw.groupby("category")["mcc_code"].agg(lambda s: s.mode()[0])
mcc_mismatch = txn_raw["mcc_code"] != txn_raw["category"].map(main_mcc)
print("операций, где mcc_code не совпадает с основным кодом категории:", mcc_mismatch.sum(), "из", len(txn_raw))

операций, где mcc_code не совпадает с основным кодом категории: 1645 из 626169


Таких операций 1645 из 626169. Это расхождение описания с фактом: код не всегда соответствует категории. По самим данным нельзя сказать, что из двух записано неверно – категория или код.

Теперь числовые поля. Начинаем с суммы операции `amount`.

In [24]:
txn_raw["amount"].describe().round(2)

count    626169.00
mean       1852.85
std        2679.21
min      -47261.19
25%         465.48
50%        1065.48
75%        2242.89
max      164155.52
Name: amount, dtype: float64

Минимум отрицательный (-47261.19). Проверяем, у каких операций сумма меньше нуля.

In [25]:
print("операций с отрицательной суммой:", (txn_raw["amount"] < 0).sum(), "из", len(txn_raw))
print("из них со статусом возврат:", ((txn_raw["amount"] < 0) & (txn_raw["status"] == "возврат")).sum())
print("всего возвратов:", (txn_raw["status"] == "возврат").sum())

операций с отрицательной суммой: 7625 из 626169
из них со статусом возврат: 7625
всего возвратов: 7625


Все 7625 отрицательных сумм – возвраты, и все возвраты записаны с минусом. Возврат уменьшает траты клиента. Еще в `amount` вместе лежат и рубли, и евро, и доллары, поэтому `describe()` по всему столбцу дает только общее представление.

По описанию `monthly_income` – ежемесячный доход в рублях.

In [26]:
clients_raw["monthly_income"].describe().round(0)

count       5204.0
mean       95083.0
std        84636.0
min         -999.0
25%        52600.0
50%        72900.0
75%       108400.0
max      1135600.0
Name: monthly_income, dtype: float64

Минимум -999, а отрицательного дохода не бывает. Смотрим подробнее на маленькие значения.

In [27]:
income = clients_raw["monthly_income"]
ranges = pd.cut(income, [-1000, 0, 1000, 10000, income.max()],
                labels=["0 и меньше", "от 0 до 1000", "от 1000 до 10000", "больше 10000"])
ranges.value_counts(sort=False, dropna=False)

monthly_income
0 и меньше            54
от 0 до 1000         172
от 1000 до 10000       0
больше 10000        4978
NaN                  486
Name: count, dtype: int64

Таблица показывает, что есть две подозрительные группы: 54 строки с доходом 0 и меньше и 172 строки с доходом меньше 1000 рублей в месяц, причем между 1000 и 10000 нет ни одного значения. Смотрим, какие значения в каждой группе.

In [28]:
print("разные значения среди дохода 0 и меньше:", income[income <= 0].unique().astype(int))

разные значения среди дохода 0 и меньше: [-999]


Во всех 54 строках доход равен ровно -999. Если бы это были ошибки ввода, числа были бы разные, а одно и то же круглое значение у всех – это условный код «нет данных». Значит это не доход, а еще один скрытый пропуск.

In [29]:
income[(income > 0) & (income < 1000)].describe().round(1)

count    172.0
mean      96.5
std       77.1
min       17.9
25%       57.3
50%       77.4
75%      111.6
max      639.1
Name: monthly_income, dtype: float64

Эти 172 значения лежат от 17.9 до 639.1, медиана 77.4. Доход 77 рублей в месяц не бывает, а медиана всего дохода 72900 рублей, то есть 72.9 тысячи. Похоже, у этих клиентов доход записан в тысячах рублей, а не в рублях, как сказано в описании. На это же указывает пустой промежуток от 1000 до 10000.

По описанию `satisfaction_score` – оценка от 1 до 5.

In [30]:
clients_raw["satisfaction_score"].value_counts(dropna=False).sort_index()

satisfaction_score
1.0     298
2.0     464
3.0     447
4.0     471
5.0     651
NaN    3359
Name: count, dtype: int64

Все оценки от 1 до 5, других значений нет. Пропусков 3359 из 5690, то есть оценки нет больше чем у половины строк.

Теперь рассмотрим даты. Форматы в описании не указаны. Разбираем каждую дату клиентов с явным форматом и считаем, сколько значений не разобралось. 

In [31]:
date_formats = {"birth_date": "%Y-%m-%d", "registration_date": "%d.%m.%Y",
                "card_open_date": "%d.%m.%Y", "app_last_login": "%Y-%m"}
for col, fmt in date_formats.items():
    d = pd.to_datetime(clients_raw[col], format=fmt, errors="coerce")
    print(col, "| не разобралось:", (d.isna() & clients_raw[col].notna()).sum(),
          "| от", d.min().date(), "до", d.max().date())

birth_date | не разобралось: 0 | от 1936-07-23 до 2002-04-27
registration_date | не разобралось: 0 | от 2010-02-16 до 2025-03-10
card_open_date | не разобралось: 0 | от 2016-01-01 до 2025-05-16
app_last_login | не разобралось: 0 | от 2024-01-01 до 2025-06-01


Все даты клиентов разобрались, но записаны в разных форматах. Все даты не позже 30.06.2025, на которую составлен справочник.

В `txn_datetime` у части операций есть время, а у части только дата. Разбираем с форматом `ISO8601`, он понимает оба варианта, и сверяем с периодом из описания.

In [32]:
txn_dt = pd.to_datetime(txn_raw["txn_datetime"], format="ISO8601", errors="coerce")
no_time = txn_raw["txn_datetime"].str.len() == 10
print("не разобралось:", txn_dt.isna().sum())
print("операций без времени:", no_time.sum(), "из", len(txn_raw))
print("самая ранняя:", txn_dt.min(), "| самая поздняя:", txn_dt.max())
print("раньше 2024-01-01:", (txn_dt < "2024-01-01").sum())
print("позже 2025-06-30:", (txn_dt >= "2025-07-01").sum())

не разобралось: 0
операций без времени: 50087 из 626169
самая ранняя: 2023-11-24 09:00:10 | самая поздняя: 2026-08-04 12:27:10
раньше 2024-01-01: 6
позже 2025-06-30: 81


Все даты разобрались. 50087 операций из 626169 записаны без времени. По описанию данные охватывают 2024-01-01 – 2025-06-30, но есть 6 операций раньше и 81 позже, вплоть до августа 2026 года, то есть позже даты, на которую составлен справочник. Описание с этим не совпадает.

Проверяем, не сосредоточены ли операции без времени в каком-то одном месяце. Доля считается от всех операций месяца.

In [33]:
(no_time.groupby(txn_dt.dt.to_period("M")).mean() * 100).round(1).loc["2024-01":"2025-06"]

txn_datetime
2024-01    7.7
2024-02    8.2
2024-03    8.2
2024-04    7.9
2024-05    8.0
2024-06    8.1
2024-07    7.8
2024-08    7.9
2024-09    8.1
2024-10    8.1
2024-11    7.9
2024-12    8.2
2025-01    7.8
2025-02    7.9
2025-03    8.0
2025-04    8.2
2025-05    7.9
2025-06    7.8
Freq: M, Name: txn_datetime, dtype: float64

В каждом месяце периода без времени от 7.7% до 8.2% операций, то есть время пропадает равномерно, а не в каком-то одном месяце. Дата у этих операций есть, поэтому для подсчета по месяцам это не мешает.

Дальше рассмотрим логику между полями. Проверяем то, что должно выполняться по смыслу: карту не могли открыть раньше прихода в банк, клиент не мог прийти в банк слишком рано по возрасту (свою карту можно получить с 14 лет), последний вход в приложение не может быть раньше регистрации, а у клиентов без приложения не должно быть даты входа.

In [34]:
birth = pd.to_datetime(clients_raw["birth_date"], format="%Y-%m-%d")
reg = pd.to_datetime(clients_raw["registration_date"], format="%d.%m.%Y")
card_open = pd.to_datetime(clients_raw["card_open_date"], format="%d.%m.%Y")
last_login = pd.to_datetime(clients_raw["app_last_login"], format="%Y-%m")
age_at_reg = (reg - birth).dt.days / 365.25

print("карта открыта раньше прихода в банк:", (card_open < reg).sum())
print("пришли в банк младше 14 лет:", (age_at_reg < 14).sum())
print("последний вход в приложение раньше месяца регистрации:", (last_login < reg.dt.to_period("M").dt.to_timestamp()).sum())
print("нет приложения, но есть дата входа:", ((clients_raw["has_mobile_app"] == "Нет") & last_login.notna()).sum())
print("есть приложение, но нет даты входа:", ((clients_raw["has_mobile_app"] == "Да") & last_login.isna()).sum())

карта открыта раньше прихода в банк: 0
пришли в банк младше 14 лет: 4
последний вход в приложение раньше месяца регистрации: 25
нет приложения, но есть дата входа: 0
есть приложение, но нет даты входа: 0


Карт, открытых раньше регистрации, нет. Но 4 клиента пришли в банк младше 14 лет, а у 25 клиентов последний вход в приложение раньше, чем они вообще стали клиентами банка. Пропуски в `app_last_login` есть только у клиентов без приложения.

По описанию `transactions.csv` – операции по картам. Но в пояснении к каналам сказано, что `СБП` – перевод по Системе быстрых платежей, а `Онлайн-банк` – операция в интернет-банке, то есть это не оплата картой. Смотрим, сколько операций в каждом канале. Доля считается от всех 626169 операций.

In [35]:
channel_counts = txn_raw["channel"].value_counts()
pd.DataFrame({"операций": channel_counts, "доля, %": (channel_counts / len(txn_raw) * 100).round(1)})

,операций,"доля, %"
channel,,
Карта,321575,51.4
СБП,107549,17.2
Токен,102000,16.3
Онлайн-банк,95045,15.2


Оплаты картой – это `Карта` (пластик) и `Токен` (телефон или часы). А `СБП` и `Онлайн-банк` занимают 17.2% и 15.2% операций, то есть в файле есть не только операции по картам, как сказано в описании.

### 2.4. Связь между таблицами

Проверяем, совпадают ли клиенты в двух таблицах. Объединяем списки `client_id` с `indicator=True`: он показывает, где нашелся клиент – только в операциях (`left_only`), только в справочнике (`right_only`) или в обеих таблицах (`both`).

In [36]:
ids = txn_raw[["client_id"]].drop_duplicates().merge(
    clients_raw[["client_id"]].drop_duplicates(), how="outer", indicator=True)
ids["_merge"].value_counts()

_merge
both          5625
left_only      365
right_only      10
Name: count, dtype: int64

У 5625 клиентов есть и операции, и запись в справочнике. 365 клиентов есть в операциях, но их нет в справочнике, а 10 клиентов из справочника не сделали ни одной операции.

Смотрим на клиентов не из справочника: сколько у них операций и когда была их последняя операция.

In [37]:
unknown_ids = ids.loc[ids["_merge"] == "left_only", "client_id"]
unknown = txn_raw["client_id"].isin(unknown_ids)
print("операций у клиентов не из справочника:", unknown.sum(), "из", len(txn_raw))
txn_dt[unknown].groupby(txn_raw.loc[unknown, "client_id"]).max().dt.to_period("Q").value_counts().sort_index()

операций у клиентов не из справочника: 32642 из 626169


txn_datetime
2024Q3      1
2024Q4    179
2025Q1    181
2025Q2      3
2025Q3      1
Freq: Q-DEC, Name: count, dtype: int64

У этих 365 клиентов 32642 операции. Последняя операция почти у всех в конце 2024 – начале 2025 года: у 179 в 4 квартале 2024 и у 181 в 1 квартале 2025. Справочник – это действующие клиенты на 30.06.2025, поэтому похоже, что это клиенты, которые ушли из банка. Операции у них есть в базовом периоде, а потом прекращаются, поэтому их уход может выглядеть как падение активности.

**Выводы по аудиту.** Перечислите найденное: что это, сколько строк,
насколько это мешает решить задачу.

Сильнее всего мешают три вещи. В операциях есть 365 клиентов, которых нет в справочнике действующих клиентов, у них 32642 операции из 626169. Последняя операция у них почти у всех в конце 2024 – начале 2025 года, поэтому похоже, что они ушли из банка, и если их не отделить, их уход будет выглядеть как падение активности. Операции через `СБП` и `Онлайн-банк` (17.2% и 15.2% от 626169 операций, вместе около трети) не являются оплатой картой, хотя файл описан как операции по картам, поэтому от выбора каналов зависит сама метрика. Наконец, 1500 операций записаны дважды – это завышает активность.

Мешают группировать, но легко исправляются: города записаны 17 способами вместо 6, категорий 12 вместо 9, у 40 строк клиентов есть полные копии, а у 15 клиентов две разные записи с разными городом или сегментом.

Касаются отдельных полей: доход пустой у 486 строк из 5690, равен -999 у 54 и похож на записанный в тысячах у 172, источник дохода `Не указан` у 1544, оценки качества нет у 3359. Это важно, только если эти поля войдут в правила групп. Еще есть операции в валюте (1283 в `EUR` и 1250 в `USD`) и операции вне периода из описания (6 раньше и 81 позже) – их мало.

Мелочи, на выводы почти не влияют: 1645 операций, где код торговой точки не совпадает с категорией, 4 клиента, которые пришли в банк младше 14 лет, и 25 клиентов со входом в приложение раньше регистрации.

## 3. Препроцессинг

На каждое изменение — строка в журнале (раздел 4). Решение «оставить
как есть» тоже является решением и тоже попадает в журнал.

### 3.1. Типы, даты, числа

Все исправления делаем на копиях, сырые данные не меняем так мы будем сравнивать, что было и что стало. Заводим журнал: каждое решение, которое меняет данные, записываем функцией `log_change`.

In [45]:
clients = clients_raw.copy()
txn = txn_raw.copy()

log_rows = []


def log_change(field, issue, scope, n_rows, value_before, action, value_after, rationale):
    log_rows.append({"field": field, "issue": issue, "scope": scope, "n_rows": n_rows,
                     "value_before": value_before, "action": action,
                     "value_after": value_after, "rationale": rationale})

Даты клиентов записаны в разных форматах. Переводим их в тип дата с теми же явными форматами, что что задали в 2.3. `app_last_login` хранит только месяц, поэтому превращается в первое число месяца.

In [ ]:
for col, fmt in date_formats.items():
    clients[col] = pd.to_datetime(clients[col], format=fmt)

log_change("birth_date, registration_date, card_open_date, app_last_login",
           "даты записаны текстом в трех разных форматах", "все строки clients", len(clients),
           "'1972-01-20', '21.03.2022', '2024-04'", "исправлено", "тип дата",
           "форматы заданы явно, все значения разобрались без ошибок")
clients[list(date_formats)].dtypes

birth_date           datetime64[us]
registration_date    datetime64[us]
card_open_date       datetime64[us]
app_last_login       datetime64[us]
dtype: object

Дату операции тоже переводим.

In [47]:
txn["txn_datetime"] = pd.to_datetime(txn["txn_datetime"], format="ISO8601")

log_change("txn_datetime", "дата записана текстом, у части операций нет времени", "все строки transactions",
           len(txn), "'2024-01-01 11:56:08', '2024-01-01'", "исправлено", "тип дата-время, без времени -> 00:00:00",
           "формат ISO8601 понимает оба варианта, все значения разобрались (проверено в 2.3)")
print("тип txn_datetime:", txn["txn_datetime"].dtype)

тип txn_datetime: datetime64[us]


Доход. Значение -999 скрытый пропуск, а не доход, поэтому заменяем его на пропуск.

In [50]:
is_code = clients["monthly_income"] == -999
clients.loc[is_code, "monthly_income"] = np.nan

log_change("monthly_income", "код -999 вместо дохода", "monthly_income == -999", is_code.sum(),
           -999, "заменено на пропуск", "NaN",
           "доход меньше нуля и одинаковый у всех 54 строк – это скрытый пропуск")
print("заменено на пропуск:", is_code.sum())

заменено на пропуск: 0


Доходы меньше 1000 рублей в месяц выглядят как записанные в тысячах: после них сразу идет большой разрыв до обычных доходов. Умножаем их на 1000 и проверяем, станут ли они похожи на доходы остальных клиентов.

In [49]:
in_thousands = (clients["monthly_income"] > 0) & (clients["monthly_income"] < 1000)
clients.loc[in_thousands, "monthly_income"] = clients.loc[in_thousands, "monthly_income"] * 1000

log_change("monthly_income", "доход записан в тысячах рублей", "0 < monthly_income < 1000", in_thousands.sum(),
           "17.9 – 639.1", "исправлено", "17900 – 639100",
           "между 1000 и 10000 нет ни одного значения, медиана 77.4 против 72900 у остальных – тот же доход в тысячах")
print("умножено на 1000:", in_thousands.sum())
clients.loc[in_thousands, "monthly_income"].describe().round(0)

умножено на 1000: 172


count       172.0
mean      96451.0
std       77055.0
min       17900.0
25%       57275.0
50%       77350.0
75%      111575.0
max      639100.0
Name: monthly_income, dtype: float64

После умножения эти доходы лежат от 17900 до 639100, медиана 77350 как и у остальных клиентов Предположение про тысячи подтверждается.

### 3.2. Категориальные признаки

### 3.3. Пропуски

Для каждого поля с пропусками: доля, предположение о механизме, решение.

### 3.4. Выбросы

Обнаружение и решение — это два разных шага. Сначала первый, потом второй.

### 3.5. Фильтры, применяемые перед расчётом активности

Покажите воронку: сколько строк на входе и сколько осталось после каждого фильтра.

## 4. Журнал изменений

Одна строка — одно решение, а не одна запись. Колонки: `field`, `issue`,
`scope`, `n_rows`, `value_before`, `action`, `value_after`, `rationale`.

In [43]:
cleaning_log = pd.DataFrame(columns=[
    "field", "issue", "scope", "n_rows",
    "value_before", "action", "value_after", "rationale",
])

## 5. Разведочный анализ

Графиков столько, сколько нужно. В отчёт пойдут не все.

### 5.1. Распределения количественных признаков

### 5.2. Частотные характеристики категориальных признаков

### 5.3. Динамика по месяцам

### 5.4. Разрезы и связи между признаками

## 6. Метрика активности

Что именно вы считаете активностью, как она вычисляется, на каком
множестве клиентов, за какие окна. Отдельно: какое снижение эта метрика
**не увидит**.

## 7. Поиск и описание групп

Правило отбора для каждой группы, её размер, причина падения.
Правило разрешения конфликтов, если клиент подходит под несколько групп.

### 7.1. Кого в кампанию не берём и почему

## 8. Что проверили и что не подтвердилось

Гипотезы, которые вы проверили и отвергли. По одной строке на гипотезу,
с числом.

## 9. Сохранение результатов

Три файла: очищенная таблица для второго ноутбука, журнал изменений,
итоговый список кампании.

In [44]:
# clean.to_parquet("clean_data.parquet", index=False)
# cleaning_log.to_csv("cleaning_log.csv", index=False, encoding="utf-8")
# campaign[["client_id", "group"]].to_csv("campaign_list.csv", index=False, encoding="utf-8")